In [1]:
import os
import torch

from torch.utils.data import DataLoader

from modules.configs.config import Config
from modules.utils.seed import set_seed

from modules.dataset.dataset import SpeakerDataset

from modules.models.ecapa_tdnn import ECAPATDNN
from modules.models.aam_softmax import AAMSoftmax

from modules.trainer import Trainer


def main():

    # --------------------------------------------------
    # Set Random Seed
    # --------------------------------------------------

    set_seed(Config.SEED)

    # --------------------------------------------------
    # Device
    # --------------------------------------------------

    device = Config.DEVICE

    print("=" * 60)
    print(f"Using Device : {device}")
    print("=" * 60)

    # --------------------------------------------------
    # Dataset
    # --------------------------------------------------

    train_dataset = SpeakerDataset(
        root_dir=Config.TRAIN_DIR,
        sample_rate=Config.SAMPLE_RATE,
        segment_length=Config.SEGMENT_LENGTH,
        train=True,
        augmentation=True
    )

    val_dataset = SpeakerDataset(
        root_dir=Config.VAL_DIR,
        sample_rate=Config.SAMPLE_RATE,
        segment_length=Config.SEGMENT_LENGTH,
        train=False,
        augmentation=False
    )

    # --------------------------------------------------
    # DataLoader
    # --------------------------------------------------

    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        num_workers=Config.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    # --------------------------------------------------
    # Number of Speakers
    # --------------------------------------------------

    num_classes = len(train_dataset.speaker_to_index)

    print(f"\nTotal Speakers : {num_classes}")
    print(f"Training Samples : {len(train_dataset)}")
    print(f"Validation Samples : {len(val_dataset)}\n")

    # --------------------------------------------------
    # Model
    # --------------------------------------------------

    model = ECAPATDNN(
        channels=Config.CHANNELS,
        scale=Config.RES2NET_SCALE,
        emb_dim=Config.EMBEDDING_SIZE
    )

    # --------------------------------------------------
    # AAM Softmax Classifier
    # --------------------------------------------------

    classifier = AAMSoftmax(
        embedding_size=Config.EMBEDDING_SIZE,
        num_classes=num_classes,
        margin=Config.MARGIN,
        scale=Config.AAM_SCALE
    )

    # --------------------------------------------------
    # Optimizer
    # --------------------------------------------------

    optimizer = torch.optim.AdamW(
        list(model.parameters()) +
        list(classifier.parameters()),
        lr=Config.LEARNING_RATE,
        weight_decay=Config.WEIGHT_DECAY
    )

    # --------------------------------------------------
    # Learning Rate Scheduler
    # --------------------------------------------------

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=Config.NUM_EPOCHS
    )

    # --------------------------------------------------
    # Trainer
    # --------------------------------------------------

    trainer = Trainer(
        model=model,
        classifier=classifier,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=Config.NUM_EPOCHS,
        checkpoint_dir=Config.CHECKPOINT_DIR
    )

    # --------------------------------------------------
    # Resume Training
    # --------------------------------------------------

    latest_checkpoint = os.path.join(
        Config.CHECKPOINT_DIR,
        "latest_model.pt"
    )

    if os.path.exists(latest_checkpoint):

        print("\nLoading latest checkpoint...\n")

        trainer.load_checkpoint(latest_checkpoint)

    else:

        print("\nNo checkpoint found. Training from scratch...\n")

    # --------------------------------------------------
    # Start Training
    # --------------------------------------------------

    history = trainer.fit()

    print("\nTraining Completed Successfully.")

    return history


if __name__ == "__main__":

    main()

IndentationError: unexpected indent (attentive_pooling.py, line 244)